In [13]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import json

# ==========================================
# 1. EXPANDED CLEANING ENGINE
# ==========================================
nltk.download('wordnet')
nltk.download('stopwords')
lemmatizer = WordNetLemmatizer()

retail_stop_words = set(stopwords.words('english'))
# Add aggressive brand and packaging filters to force consolidation
retail_stop_words.update([
    'organic', 'bag', 'of', 'large', 'small', 'oz', 'pack', 'fresh', 'bunch', 'free', 'gluten',
    'sabra', 'cedars', 'hope', 'stacy', 'kraft', 'chobani', 'trader', 'joes', 'whole', 'foods',
    'value', 'great', 'classic', 'original', 'traditional', 'style', 'size', 'lb', 'ct', 'count'
])

def clean_text(text):
    original = str(text)
    text = original.lower()
    # Remove punctuation, symbols, and dimensions/numbers (e.g., 10oz, 4ct)
    text = re.sub(r'\d+oz|\d+ct|\d+lb|\d+|\b\d+\b', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    words = [lemmatizer.lemmatize(word) for word in text.split() if word not in retail_stop_words]
    cleaned = " ".join(words).strip()
    
    if not cleaned:
        return re.sub(r'[^a-z0-9\s]', '', original.lower()).strip()
    return cleaned

print("Executing Lexical Deduplication...")
unique_products = pd.DataFrame({'original_name': merged_df['product_name'].unique()})
unique_products['cleaned_name'] = unique_products['original_name'].apply(clean_text)

# ==========================================
# 2. EXACT-MATCH SOURCE CONSOLIDATION
# ==========================================
# This ensures that "Hummus, Original" and "Original Hummus" instantly become identical 
# before any machine learning algorithms are executed.
print("Consolidating identical text mappings...")
item_counts = merged_df['product_name'].value_counts().reset_index()
item_counts.columns = ['original_name', 'purchase_count']
unique_products = pd.merge(unique_products, item_counts, on='original_name')

# Map each cleaned string variant to the absolute most popular original name
most_popular_variants = unique_products.loc[unique_products.groupby('cleaned_name')['purchase_count'].idxmax()]
exact_match_map = dict(zip(most_popular_variants['cleaned_name'], most_popular_variants['original_name']))

# Assign the canonical exact match name
unique_products['canonical_group'] = unique_products['cleaned_name'].map(exact_match_map)

# ==========================================
# 3. K-MEANS CLUSTERING (FOR WIDER CATEGORIES)
# ==========================================
print("Vectorizing for secondary cluster matching...")
# Pull unique canonical names to run clustering on a clean, condensed list
canonical_catalog = pd.DataFrame({'cleaned_name': unique_products['cleaned_name'].unique()})

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=1)
X = vectorizer.fit_transform(canonical_catalog['cleaned_name'])

print("Clustering broad product classes...")
# Reduced cluster count to 120 to force broader grouping (e.g., merging near-similar items)
cluster_model = MiniBatchKMeans(n_clusters=120, batch_size=1024, random_state=42, n_init='auto')
canonical_catalog['cluster_id'] = cluster_model.fit_predict(X)

# Re-map back to the primary counts to find the overall cluster representative
cluster_rep_lookup = pd.merge(unique_products, canonical_catalog, on='cleaned_name')
top_cluster_idx = cluster_rep_lookup.groupby('cluster_id')['purchase_count'].idxmax()
cluster_representatives = cluster_rep_lookup.loc[top_cluster_idx, ['cluster_id', 'canonical_group']]
cluster_representatives.rename(columns={'canonical_group': 'representative_name'}, inplace=True)

# Build the definitive dictionary for the Web Backend
final_mapping_df = pd.merge(cluster_rep_lookup, cluster_representatives, on='cluster_id')
group_to_specific_map = dict(zip(final_mapping_df['original_name'], final_mapping_df['representative_name']))

with open('../app/group_mapping.json', 'w') as f:
    json.dump(group_to_specific_map, f)

# ==========================================
# 4. TRANSACTION MINING & FP-GROWTH
# ==========================================
print("Preparing transaction arrays...")
merged_df['product_group'] = merged_df['product_name'].map(group_to_specific_map)

top_groups = merged_df['product_group'].value_counts().head(1500).index
filtered_df = merged_df[merged_df['product_group'].isin(top_groups)]

sample_order_ids = filtered_df['order_id'].drop_duplicates().sample(n=300000, random_state=42)
final_df = filtered_df[filtered_df['order_id'].isin(sample_order_ids)]

transactions = final_df.groupby('order_id')['product_group'].apply(list).values

print("Running FP-Growth Engine...")
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket_sets = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = fpgrowth(basket_sets, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Deduplicate self-referencing rules (e.g., Hummus -> Hummus)
rules['antecedents'] = rules['antecedents'].apply(lambda x: list(x)[0] if len(x) > 0 else None)
rules['consequents'] = rules['consequents'].apply(lambda x: list(x)[0] if len(x) > 0 else None)
rules = rules.dropna(subset=['antecedents', 'consequents'])

# CRITICAL FILTER: Strip out any rule where the item recommends itself
rules = rules[rules['antecedents'] != rules['consequents']]

final_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)
final_rules.to_csv('../app/rules.csv', index=False)

print(f"Data pipeline complete! Exported {len(final_rules)} clean, highly distinct rules.")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\anton\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anton\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Executing Lexical Deduplication...
Consolidating identical text mappings...
Vectorizing for secondary cluster matching...
Clustering broad product classes...
Preparing transaction arrays...
Running FP-Growth Engine...
Data pipeline complete! Exported 14344 clean, highly distinct rules.
